# Create a Wide-Area Ethernet (Layer 2) Network: Manual Configuration

This notebook shows how to create an isolated local Ethernet and connect compute nodes to it.  

*The original code was developed by the FABRIC team: jupyter-examples-rel1.9.0/fabric_examples/fablib_api/create_l2network_wide_area/create_l2network_wide_area_manual.ipynb*



## Import the FABlib Library


In [1]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

User: praveen.rao@missouri.edu bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,23.134.232.211
Token File,/home/fabric/.tokens.json
Project ID,309afbc6-6f04-4eca-b56d-1d29a1c4b3da
Bastion Host,bastion.fabric-testbed.net
Bastion Username,praveen_rao_0051543462
Bastion Private Key File,/home/fabric/work/fabric_config/nsfgafbastion
Slice Public Key File,/home/fabric/work/fabric_config/nsfgaf.pub


## Create the Experiment Slice

The following creates two nodes with basic NICs connected to an isolated WAN Ethernet.  

Two nodes are created and one NIC component is added to each node.  This example uses components of model `NIC_Basic` which are SR-IOV Virtual Function on a 100 Gpbs Mellanox ConnectX-6 PCI device. The VF is accessed by the node via PCI passthrough. Other NIC models are listed below. When using dedicated PCI devices the whole physical device is allocated to one node and the device is accessed by the node using PCI passthrough. Calling the `get_interfaces()` method on a component will return a list of interfaces. Many dedicated NIC components may have more than one port.  Either port can be connected to the network.

Next, add an `l2network` to the slice and pass the list of interfaces you want connected to this Ethernet. If the interfaces in the list are located on two sites, the network will automatically create a wide-area layer 2 circuit.  By default, a node is put on a random site.  If you want to ensure that your nodes are all on different sites you can specify the name of the sites in the `add_node` methode.  You can use the `fablib.get_random_site()` method to get a set of random site names that guarantee that the sites are different. 

Manual configuration does not require any additional steps before the slice request is submitted.

NIC component models options:
- NIC_Basic: 100 Gbps Mellanox ConnectX-6 SR-IOV VF (1 Port)
- NIC_ConnectX_5: 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device (2 Ports) 
- NIC_ConnectX_6: 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device (2 Ports) 

In [2]:
try:
    fablib.list_sites()
except Exception as e:
    print(f"Exception: {e}")

Name,State,Address,Location,PTP Capable,Hosts,CPUs,Cores Available,Cores Capacity,Cores Allocated,Ram Available,Ram Capacity,Ram Allocated,Disk Available,Disk Capacity,Disk Allocated,Basic NIC Available,Basic NIC Capacity,Basic NIC Allocated,P4-Switch Available,P4-Switch Capacity,P4-Switch Allocated,ConnectX-6 Available,ConnectX-6 Capacity,ConnectX-6 Allocated,ConnectX-5 Available,ConnectX-5 Capacity,ConnectX-5 Allocated,ConnectX-7-100 Available,ConnectX-7-100 Capacity,ConnectX-7-100 Allocated,ConnectX-7-400 Available,ConnectX-7-400 Capacity,ConnectX-7-400 Allocated,BlueField2-ConnectX-6 Available,BlueField2-ConnectX-6 Capacity,BlueField2-ConnectX-6 Allocated,NVMe Available,NVMe Capacity,NVMe Allocated,Tesla T4 Available,Tesla T4 Capacity,Tesla T4 Allocated,RTX6000 Available,RTX6000 Capacity,RTX6000 Allocated,A30 Available,A30 Capacity,A30 Allocated,A40 Available,A40 Capacity,A40 Allocated,U280 Available,U280 Capacity,U280 Allocated,SN1022 Available,SN1022 Capacity,SN1022 Allocated
UCSD,Active,"10100 Hopkins Drive,CA 92093","(32.8886802, -117.239324)",True,5,10,502,640,138,2034,2390,356,53727,58717,4990,535,635,100,1,1,0,1,2,1,4,4,0,0,0,0,0,0,0,0,0,0,16,16,0,4,4,0,6,6,0,0,0,0,0,0,0,1,1,0,0,0,0
TACC,Active,"10100 Burnet Rd,Austin, TX 78758","(30.3899405, -97.7261807)",False,5,10,204,640,436,598,2390,1792,100563,107463,6900,600,635,35,0,0,0,2,2,0,4,4,0,1,1,0,0,0,0,0,0,0,16,16,0,0,4,4,0,6,6,0,0,0,0,0,0,1,1,0,0,0,0
CLEM,Active,"340 Computer Court,Anderson,SC 29625","(34.5865435, -82.8212889)",True,3,6,272,384,112,1162,1434,272,53939,56959,3020,376,381,5,0,0,0,2,2,0,2,2,0,0,0,0,0,0,0,0,0,0,10,10,0,1,2,1,1,3,2,0,0,0,0,0,0,1,1,0,0,0,0
NCSA,Active,"1725 S Oak St.,Champaign, IL 61820","(40.09584, -88.2415369)",True,3,6,76,384,308,738,1434,696,60441,64161,3720,364,381,17,0,0,0,0,2,2,1,2,1,1,1,0,0,0,0,0,0,0,10,10,0,0,2,2,3,3,0,0,0,0,0,0,0,1,1,0,0,0,0
RUTG,Active,"120 Avenue E,Piscataway, New Jersey","(40.5224962, -74.4405719)",True,5,10,506,640,134,1854,2390,536,106633,107463,830,624,635,11,0,0,0,2,2,0,4,4,0,1,1,0,0,0,0,0,0,0,16,16,0,0,0,0,0,0,0,0,8,8,0,0,0,1,1,0,0,0,0
KANS,Active,"1100 Walnut Street,Kansas City,MO 64106","(39.1004885, -94.5823448)",False,3,6,90,384,294,642,1434,792,60271,64161,3890,345,381,36,0,0,0,1,2,1,2,2,0,1,1,0,0,0,0,0,0,0,9,10,1,0,0,0,0,0,0,0,4,4,0,0,0,1,1,0,0,0,0
SALT,Active,"572 Delong Street,Salt Lake City, UT 84104","(40.7570751, -111.9534664)",False,3,6,314,384,70,1142,1410,268,62651,64161,1510,372,381,9,0,0,0,1,2,1,2,2,0,0,0,0,1,1,0,0,0,0,10,10,0,2,2,0,2,3,1,0,0,0,0,0,0,0,1,1,0,0,0
FIU,Active,"11001 SW 14th St,Miami,FL 33199","(25.7542948, -80.3702894)",True,5,10,498,640,142,2142,2390,248,55603,58713,3110,625,635,10,1,1,0,2,2,0,4,4,0,0,1,1,0,0,0,0,0,0,16,16,0,4,4,0,2,6,4,0,0,0,0,0,0,1,1,0,0,0,0
MICH,Active,"2530 Draper Dr,Ann Arbor, MI 48109","(42.2931086, -83.7101319)",True,3,6,146,384,238,858,1434,576,61741,64161,2420,369,381,12,1,1,0,1,2,1,1,2,1,1,1,0,0,0,0,0,0,0,10,10,0,0,2,2,3,3,0,0,0,0,0,0,0,1,1,0,0,0,0
INDI,Active,"535 West Michigan Street,Indianapolis, IN 46202","(39.7737312, -86.1674868)",False,3,6,144,384,240,978,1434,456,53118,56618,3500,365,381,16,0,0,0,0,2,2,2,2,0,0,0,0,0,0,0,0,0,0,10,10,0,0,0,0,0,0,0,4,4,0,0,0,0,1,1,0,0,0,0


In [23]:
try:
    fablib.list_sites(fields=['name','address', 'nic_connectx_5_available', 'nic_connectx_6_available', 'tesla_t4_available', 'rtx6000_available', 'a30_available', 'a40_available'])
except Exception as e:
    print(f"Exception: {e}")

Name,Address,ConnectX-5 Available,ConnectX-6 Available,Tesla T4 Available,RTX6000 Available,A30 Available,A40 Available
SALT,"572 Delong Street,Salt Lake City, UT 84104",2,1,2,2,0,0
KANS,"1100 Walnut Street,Kansas City,MO 64106",2,1,0,0,2,0
NEWY,"32 Sixth Avenue,New York, NY 10013",0,1,0,0,0,0
CERN,"Meyrin site, Auvergne-Rhone-Alpes, Metropolitan France, 01280, France",4,1,0,0,2,0
ATLA,"180 Peachtree,Atlanta, GA 30303",3,0,0,0,0,0
AMST,"Science Park 904, 1098 XH Amsterdam",2,3,0,0,2,0
STAR,"710 North Lakeshore Drive, 60611",3,0,6,1,0,0
EDUKY,"301 Hilltop Avenue,Lexington, KY 40506",0,0,0,0,0,0
UCSD,"10100 Hopkins Drive,CA 92093",2,1,3,3,0,0
GATECH,"760 West Peachtree Street NW,Atlanta, GA 30308",4,1,0,0,2,0


# There are multiple ways to stitch GPUs across two sites

# If you want two VMs spanning two sites with few GPUs each, use the below cell

In [29]:
slice_name = 'ST-UC'

# Select the sites
site1 = "STAR"
site2 = "DALL"
print(f"Sites: {site1}, {site2}") # Approach of add_l2network works only for two unique sites

node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

#Create Slice
slice = fablib.new_slice(name=slice_name)

# Cores, RAM, disk space
num_cores=12
RAM_size=32
disk_size=100
OS_image='default_ubuntu_20'
g1='GPU_TeslaT4'
g2='GPU_RTX6000'
g3='GPU_A30'

# Network - CREATE this first to avoid errors with multiple interfaces for L2STS
net1 = slice.add_l2network(name=network_name)

# Node1

node1 = slice.add_node(name=node1_name, site=site1, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]
iface12 = node1.add_component(model='NIC_ConnectX_5', name=node1_nic_name+"2").get_interfaces()[0]
net1.add_interface(iface1)
net1.add_interface(iface12)

# GPUs
node1.add_component(model=g1, name="a1")
node1.add_component(model=g1, name="a2")
#node1.add_component(model=g2, name="a3")
#node1.add_component(model=g2, name="a4")

# Node2
node2 = slice.add_node(name=node2_name, site=site2, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface2 = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]
iface22 = node2.add_component(model='NIC_ConnectX_5', name=node2_nic_name+"2").get_interfaces()[0]
net1.add_interface(iface2)
net1.add_interface(iface22)

# GPUs
node2.add_component(model=g1, name="b1")
node2.add_component(model=g1, name="b2")
#node2.add_component(model=g2, name="b3")
#node2.add_component(model=g2, name="b4")


#Submit Slice Request
slice.submit();


Retry: 11, Time: 285 sec


ID,40f65b93-388d-4c6f-9741-cb1d1ed61780
Name,ST-UC
Lease Expiration (UTC),2026-02-14 17:05:19 +0000
Lease Start (UTC),2026-02-13 17:05:19 +0000
Project ID,309afbc6-6f04-4eca-b56d-1d29a1c4b3da
State,StableOK
Email,praveen.rao@missouri.edu
UserId,9213b748-d6e5-4daf-9758-525629679049


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
069e7c2d-79c1-4730-8077-884ee236e894,Node1,12,32,100,default_ubuntu_20,qcow2,star-w4.fabric-testbed.net,STAR,ubuntu,2001:400:a100:3030:f816:3eff:fe30:3122,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3030:f816:3eff:fe30:3122,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
3f57a3d8-80fe-4e2c-a7d4-427e00faac31,Node2,12,32,100,default_ubuntu_20,qcow2,dall-w3.fabric-testbed.net,DALL,ubuntu,2001:400:a100:3000:f816:3eff:fe9a:4866,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3000:f816:3eff:fe9a:4866,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
752ecb31-7219-440f-b06e-66bd761e2611,net1,L2,L2STS,None,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic12-p1,p1,Node1,None,25,config,,0C:42:A1:94:3A:60,enp7s0,enp7s0,None,1,None
Node1-nic12-p2,p2,Node1,net1,25,config,,0C:42:A1:94:3A:61,enp8s0,enp8s0,fe80::e42:a1ff:fe94:3a61,1,TwentyFiveGigE0/0/0/16/1
Node1-nic1-p1,p1,Node1,net1,100,config,,06:FC:6B:FF:76:34,enp11s0,enp11s0,fe80::4fc:6bff:feff:7634,4,HundredGigE0/0/0/11
Node2-nic2-p1,p1,Node2,net1,100,config,,16:DA:3E:A6:B6:00,enp9s0,enp9s0,fe80::14da:3eff:fea6:b600,4,HundredGigE0/0/0/9
Node2-nic22-p1,p1,Node2,net1,25,config,,B8:CE:F6:AF:83:42,enp10s0,enp10s0,fe80::bace:f6ff:feaf:8342,1,TwentyFiveGigE0/0/0/10/0
Node2-nic22-p2,p2,Node2,None,25,config,,B8:CE:F6:AF:83:43,enp11s0,enp11s0,None,1,None



Time to print interfaces 297 seconds


In [3]:
slice_name="STAR-UCSD"
slice = fablib.get_slice(name=slice_name)
slice.show()
slice.list_nodes()
slice.list_networks()
slice.list_interfaces()

ID,46bd0bd7-3aff-4b78-a326-8b8c9b5db0ec
Name,STAR-UCSD
Lease Expiration (UTC),2026-02-22 15:50:39 +0000
Lease Start (UTC),2026-02-13 15:50:39 +0000
Project ID,309afbc6-6f04-4eca-b56d-1d29a1c4b3da
State,StableOK
Email,praveen.rao@missouri.edu
UserId,9213b748-d6e5-4daf-9758-525629679049


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
d47cf51f-2b3b-478e-b75f-18e42235ba08,Node1,12,32,500,default_ubuntu_20,qcow2,max-w4.fabric-testbed.net,MAX,ubuntu,2001:468:c00:ffc4:f816:3eff:fef2:404e,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:468:c00:ffc4:f816:3eff:fef2:404e,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
a40fd2d7-a337-4c0f-a9a9-d21b75490f79,Node2,12,32,500,default_ubuntu_20,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe8d:f06b,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe8d:f06b,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
d9645704-8af2-453a-bfdc-a45afb136bdb,net1,L2,L2STS,None,net1.subnet,net1.gateway,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node2-nic22-p1,p1,Node2,net1,25,config,,10:70:FD:C1:59:BC,enp10s0,enp10s0,192.168.1.4,1,TwentyFiveGigE0/0/0/18/0
Node2-nic22-p2,p2,Node2,None,25,config,,10:70:FD:C1:59:BD,enp11s0,enp11s0,None,1,None
Node2-nic2-p1,p1,Node2,net1,100,config,,0A:3D:4E:3E:91:96,enp9s0,enp9s0,192.168.1.3,4,HundredGigE0/0/0/13
Node1-nic1-p1,p1,Node1,net1,100,config,,02:E1:92:DA:C9:05,enp11s0,enp11s0,192.168.1.1,4,HundredGigE0/0/0/11
Node1-nic12-p1,p1,Node1,net1,25,config,,04:3F:72:FA:75:60,enp9s0,enp9s0,192.168.1.2,6,TwentyFiveGigE0/0/0/16/2
Node1-nic12-p2,p2,Node1,None,25,config,,04:3F:72:FA:75:61,enp10s0,enp10s0,None,6,None


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node2-nic22-p1,p1,Node2,net1,25,config,,10:70:FD:C1:59:BC,enp10s0,enp10s0,192.168.1.4,1,TwentyFiveGigE0/0/0/18/0
Node2-nic22-p2,p2,Node2,None,25,config,,10:70:FD:C1:59:BD,enp11s0,enp11s0,None,1,None
Node2-nic2-p1,p1,Node2,net1,100,config,,0A:3D:4E:3E:91:96,enp9s0,enp9s0,192.168.1.3,4,HundredGigE0/0/0/13
Node1-nic1-p1,p1,Node1,net1,100,config,,02:E1:92:DA:C9:05,enp11s0,enp11s0,192.168.1.1,4,HundredGigE0/0/0/11
Node1-nic12-p1,p1,Node1,net1,25,config,,04:3F:72:FA:75:60,enp9s0,enp9s0,192.168.1.2,6,TwentyFiveGigE0/0/0/16/2
Node1-nic12-p2,p2,Node1,None,25,config,,04:3F:72:FA:75:61,enp10s0,enp10s0,None,6,None


# FABnet L3 Network

In [11]:
slice_name = 'STAR-STAR'

# Select the sites
site1 = "STAR"
site2 = "STAR"
print(f"Sites: {site1}, {site2}") # Approach of add_l2network works only for two unique sites

node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

#Create Slice
slice = fablib.new_slice(name=slice_name)

# Cores, RAM, disk space
num_cores=12
RAM_size=32
disk_size=100
OS_image='default_ubuntu_20'
g1='GPU_TeslaT4'
g2='GPU_RTX6000'
g3='GPU_A30'

# Node1

node1 = slice.add_node(name=node1_name, site=site1, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface1A = node1.add_component(model='NIC_Basic', name=node1_nic_name+"A").get_interfaces()[0]
iface1B = node1.add_component(model='NIC_Basic', name=node1_nic_name+"B").get_interfaces()[0]
# GPUs
node1.add_component(model=g2, name="a1")
node1.add_component(model=g2, name="a2")
#node1.add_component(model=g2, name="a3")
#node1.add_component(model=g2, name="a4")

# Node2
node2 = slice.add_node(name=node2_name, site=site2, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface2A = node2.add_component(model='NIC_Basic', name=node2_nic_name+"A").get_interfaces()[0]
iface2B = node2.add_component(model='NIC_Basic', name=node2_nic_name+"B").get_interfaces()[0]
# GPUs
node2.add_component(model=g1, name="b1")
node2.add_component(model=g1, name="b2")
#node2.add_component(model=g2, name="b3")
#node2.add_component(model=g2, name="b4")

# Network
net1 = slice.add_l3network(name=network_name+"1", interfaces=[iface1A, iface1B], type='IPv4')
net2 = slice.add_l3network(name=network_name+"2", interfaces=[iface2A, iface2B], type='IPv4')


#Submit Slice Request
slice.submit();


Retry: 1, Time: 43 sec


ID,1f0df59a-fafb-4ab7-b4db-289c7e6f3723
Name,STAR-STAR
Lease Expiration (UTC),2026-02-13 16:42:49 +0000
Lease Start (UTC),2026-02-12 16:42:49 +0000
Project ID,309afbc6-6f04-4eca-b56d-1d29a1c4b3da
State,Closing
Email,praveen.rao@missouri.edu
UserId,9213b748-d6e5-4daf-9758-525629679049


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
4c08d4a9-b63c-4c26-93d2-d291157f2011,Node1,None,None,None,default_ubuntu_20,qcow2,None,STAR,ubuntu,,Closed,Insufficient resources : [core]#,,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
0b8fc928-246d-4fee-81bf-0e2b8602e3c9,Node2,12,32,100,default_ubuntu_20,qcow2,star-w6.fabric-testbed.net,STAR,ubuntu,,Closed,TicketReviewPolicy: Closing reservation due to failure in slice#,,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
ad545932-f008-40fb-a386-596955163924,net11,L3,FABNetv4,STAR,net11.subnet,net11.gateway,Closed,"redeem predecessor reservation# 4c08d4a9-b63c-4c26-93d2-d291157f2011 is in a terminal state, failing the reservation# ad545932-f008-40fb-a386-596955163924#"
93b6ad3e-52c4-445e-8a5c-2b9a0c2f5224,net12,L3,FABNetv4,STAR,net12.subnet,net12.gateway,Closed,TicketReviewPolicy: Closing reservation due to failure in slice#


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1B-p1,p1,Node1,net11,100,config,,,,,,,p1
Node1-nic1A-p1,p1,Node1,net11,100,config,,,,,,,p1
Node2-nic2B-p1,p1,Node2,net12,100,config,,,,,,,HundredGigE0/0/0/3
Node2-nic2A-p1,p1,Node2,net12,100,config,,,,,,,HundredGigE0/0/0/3



Time to print interfaces 44 seconds


# If you want 2 VMs per site across two sites, try the below cell

In [11]:
slice_name = 'GIVE-A-NAME'

# Select the sites
site1 = "UCSD"
site2 = "STAR"
print(f"Sites: {site1}, {site2}") # Approach of add_l2network works only for two unique sites

node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

#Create Slice
slice = fablib.new_slice(name=slice_name)

# Cores, RAM, disk space
num_cores=12
RAM_size=32
disk_size=500
OS_image='default_ubuntu_20'
g1='GPU_TeslaT4'
g2='GPU_RTX6000'
g3='GPU_A30'

#https://learn.fabric-testbed.net/forums/topic/create-layer-2-interface-amount-5-nodes/ -- USEFUL reference

# Node1

node1A = slice.add_node(name=node1_name+"A", site=site1, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface1A = node1A.add_component(model='NIC_Basic', name=node1_nic_name+"A").get_interfaces()[0]
iface1A2 = node1A.add_component(model='NIC_Basic', name=node1_nic_name+"A2").get_interfaces()[0]
# GPUs
node1A.add_component(model=g2, name="a1A")
node1A.add_component(model=g2, name="a2A")
# node1.add_component(model=g2, name="a3")
# node1.add_component(model=g2, name="a4")

node1B = slice.add_node(name=node1_name+"B", site=site1, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface1B = node1B.add_component(model='NIC_Basic', name=node1_nic_name+"B").get_interfaces()[0]
# GPUs
node1B.add_component(model=g2, name="a1B")
node1B.add_component(model=g2, name="a2B")
# node1.add_component(model=g2, name="a3")
# node1.add_component(model=g2, name="a4")


# Node2
node2A = slice.add_node(name=node2_name+"A", site=site2, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface2A = node2A.add_component(model='NIC_Basic', name=node2_nic_name+"A").get_interfaces()[0]
iface2A2 = node2A.add_component(model='NIC_Basic', name=node2_nic_name+"A2").get_interfaces()[0]
# GPUs
node2A.add_component(model=g1, name="b1A")
node2A.add_component(model=g1, name="b2A")
# node2.add_component(model=g1, name="b3")
# node2.add_component(model=g1, name="b4")

# Node2
node2B = slice.add_node(name=node2_name+"B", site=site2, cores=num_cores, ram=RAM_size, disk=disk_size, image=OS_image)
iface2B = node2B.add_component(model='NIC_Basic', name=node2_nic_name+"B").get_interfaces()[0]
# GPUs
node2B.add_component(model=g1, name="b1B")
node2B.add_component(model=g1, name="b2B")
# node2.add_component(model=g1, name="b3")
# node2.add_component(model=g1, name="b4")

# Network
net1 = slice.add_l2network(name=network_name+"A", interfaces=[iface1A, iface1B])
net2 = slice.add_l2network(name=network_name+"B", interfaces=[iface2A, iface2B])
net3 = slice.add_l2network(name=network_name+"C", interfaces=[iface1A2, iface2A2])

#Submit Slice Request
slice.submit();


Retry: 15, Time: 413 sec


ID,42a930ce-79e0-4043-8d60-15c2bfdde6af
Name,UC-STAR
Lease Expiration (UTC),2026-01-16 21:34:24 +0000
Lease Start (UTC),2026-01-15 21:34:24 +0000
Project ID,309afbc6-6f04-4eca-b56d-1d29a1c4b3da
State,StableOK
Email,praveen.rao@missouri.edu
UserId,9213b748-d6e5-4daf-9758-525629679049


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
49d8e3d0-70e1-473d-8a2a-d71abcc69d85,Node1A,12,32,500,default_ubuntu_20,qcow2,ucsd-w2.fabric-testbed.net,UCSD,ubuntu,2001:48d0:6031:1991:f816:3eff:fe46:dedc,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48d0:6031:1991:f816:3eff:fe46:dedc,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
6a34e5a7-cebe-4463-9779-67057ed663d3,Node1B,12,32,500,default_ubuntu_20,qcow2,ucsd-w1.fabric-testbed.net,UCSD,ubuntu,2001:48d0:6031:1991:f816:3eff:fe2e:6188,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48d0:6031:1991:f816:3eff:fe2e:6188,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
9984ff74-c3b2-4fbc-ae91-90d897700f89,Node2A,12,32,500,default_ubuntu_20,qcow2,star-w6.fabric-testbed.net,STAR,ubuntu,2001:400:a100:3030:f816:3eff:fe1f:ac27,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3030:f816:3eff:fe1f:ac27,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
c90afa74-c6d0-4299-9d22-2a256d50f7e2,Node2B,12,32,500,default_ubuntu_20,qcow2,star-w4.fabric-testbed.net,STAR,ubuntu,2001:400:a100:3030:f816:3eff:fefb:2d4f,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3030:f816:3eff:fefb:2d4f,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
6b5c5ba5-7814-47da-9cf2-afe14349439f,net1A,L2,L2Bridge,UCSD,None,None,Active,
06b1b7aa-aee4-47f4-ba0a-ac5103d5edf0,net1B,L2,L2Bridge,STAR,None,None,Active,
a915ec69-6b07-404b-98b2-933d4ccfc7e1,net1C,L2,L2STS,None,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1A-nic1A-p1,p1,Node1A,net1A,100,config,,BE:78:F8:4F:D3:2E,enp7s0,enp7s0,fe80::bc78:f8ff:fe4f:d32e,6,HundredGigE0/0/0/7
Node1A-nic1A2-p1,p1,Node1A,net1C,100,config,,BE:7F:CB:5F:88:CD,enp9s0,enp9s0,fe80::bc7f:cbff:fe5f:88cd,6,HundredGigE0/0/0/7
Node1B-nic1B-p1,p1,Node1B,net1A,100,config,,1A:76:ED:CE:CD:C4,enp8s0,enp8s0,fe80::1876:edff:fece:cdc4,6,HundredGigE0/0/0/5
Node2A-nic2A-p1,p1,Node2A,net1B,100,config,,02:65:F1:2D:1E:A8,enp8s0,enp8s0,fe80::65:f1ff:fe2d:1ea8,4,HundredGigE0/0/0/3
Node2A-nic2A2-p1,p1,Node2A,net1C,100,config,,02:A0:E3:47:AB:A5,enp10s0,enp10s0,fe80::a0:e3ff:fe47:aba5,4,HundredGigE0/0/0/3
Node2B-nic2B-p1,p1,Node2B,net1B,100,config,,02:67:7F:D4:71:C5,enp9s0,enp9s0,fe80::67:7fff:fed4:71c5,4,HundredGigE0/0/0/11



Time to print interfaces 428 seconds


In [9]:
slice_name="ST-UC"
slice = fablib.get_slice(name=slice_name)
slice.show()
slice.list_nodes()
slice.list_networks()
slice.list_interfaces()

ID,0e965e53-df96-4d7f-9681-90cd02c68b05
Name,STAR-STAR
Lease Expiration (UTC),2026-02-15 22:26:02 +0000
Lease Start (UTC),2026-02-10 22:26:02 +0000
Project ID,309afbc6-6f04-4eca-b56d-1d29a1c4b3da
State,StableOK
Email,praveen.rao@missouri.edu
UserId,9213b748-d6e5-4daf-9758-525629679049


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
5f0e12a4-4572-4c89-aa4c-d1ef8f31878e,Node1,12,16,100,default_ubuntu_20,qcow2,star-w6.fabric-testbed.net,STAR,ubuntu,2001:400:a100:3030:f816:3eff:fef1:87a,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3030:f816:3eff:fef1:87a,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf
98af2bc5-4f30-4d20-aebf-be5266bc9af2,Node2,12,16,100,default_ubuntu_20,qcow2,star-w5.fabric-testbed.net,STAR,ubuntu,2001:400:a100:3030:f816:3eff:febc:1df1,Active,,ssh -i /home/fabric/work/fabric_config/nsfgaf -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3030:f816:3eff:febc:1df1,/home/fabric/work/fabric_config/nsfgaf.pub,/home/fabric/work/fabric_config/nsfgaf


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
faa00325-8d73-4480-8f88-8dea50ccb8cb,net1,L2,L2Bridge,STAR,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic12-p1,p1,Node1,net1,25,config,,0C:42:A1:94:3B:C4,enp9s0,enp9s0,fe80::e42:a1ff:fe94:3bc4,6,TwentyFiveGigE0/0/0/20/2
Node1-nic12-p2,p2,Node1,None,25,config,,0C:42:A1:94:3B:C5,enp10s0,enp10s0,None,6,None
Node1-nic1-p1,p1,Node1,net1,100,config,,3A:DD:F6:0F:A3:23,enp8s0,enp8s0,fe80::38dd:f6ff:fe0f:a323,4,HundredGigE0/0/0/3
Node2-nic22-p1,p1,Node2,net1,25,config,,0C:42:A1:94:3B:CC,enp7s0,enp7s0,fe80::e42:a1ff:fe94:3bcc,6,TwentyFiveGigE0/0/0/19/2
Node2-nic22-p2,p2,Node2,None,25,config,,0C:42:A1:94:3B:CD,enp8s0,enp8s0,None,6,None
Node2-nic2-p1,p1,Node2,net1,100,config,,0E:F2:BC:35:CF:DE,enp10s0,enp10s0,fe80::cf2:bcff:fe35:cfde,4,HundredGigE0/0/0/1


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic12-p1,p1,Node1,net1,25,config,,0C:42:A1:94:3B:C4,enp9s0,enp9s0,fe80::e42:a1ff:fe94:3bc4,6,TwentyFiveGigE0/0/0/20/2
Node1-nic12-p2,p2,Node1,None,25,config,,0C:42:A1:94:3B:C5,enp10s0,enp10s0,None,6,None
Node1-nic1-p1,p1,Node1,net1,100,config,,3A:DD:F6:0F:A3:23,enp8s0,enp8s0,fe80::38dd:f6ff:fe0f:a323,4,HundredGigE0/0/0/3
Node2-nic22-p1,p1,Node2,net1,25,config,,0C:42:A1:94:3B:CC,enp7s0,enp7s0,fe80::e42:a1ff:fe94:3bcc,6,TwentyFiveGigE0/0/0/19/2
Node2-nic22-p2,p2,Node2,None,25,config,,0C:42:A1:94:3B:CD,enp8s0,enp8s0,None,6,None
Node2-nic2-p1,p1,Node2,net1,100,config,,0E:F2:BC:35:CF:DE,enp10s0,enp10s0,fe80::cf2:bcff:fe35:cfde,4,HundredGigE0/0/0/1


## Delete the slice

In [10]:
slice = fablib.get_slice(name=slice_name)
slice.delete()

## Manually Configure IP Addresses

Some experiments use FABRIC layer 2 networks to enable deploying non-IP layer 3 networks.  If this describes your exepriment, your nodes and network are ready. You can now login to the nodes and deploy your experiemnt.

Most users will want to configure IP addresses on there new nodes.  FABlib provides some useful methods to help you configure basic IP addreses. 

### Pick a Subnet

Create subnet and list of available IP addresses. All object are Python IP managment objects. You can use either IPv4 or IPv6 subents and addresses.

In [ ]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network

subnet = IPv4Network("192.168.1.0/24")
available_ips = list(subnet)[1:]

### Configure Node1

Get the node and the interface you wish to configure.  You can use `node.get_interface` to get the interface that is connected to the specified network.  Then `pop` an IP address from the list of available IPs and call `iface.ip_addr_add` to set the IP and subnet.  

Optionally, use the `node.execute()` method to show the results of adding the IP address.

In [ ]:
node1 = slice.get_node(name=node1_name)        
node1_iface = node1.get_interface(network_name=network_name) 
node1_addr = available_ips.pop(0)
node1_iface.ip_addr_add(addr=node1_addr, subnet=subnet)

stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_device_name()}')

### Configure Node2

Repeat the steps to add the next available IP to the second node.

In [ ]:
node2 = slice.get_node(name=node2_name)        
node2_iface = node2.get_interface(network_name=network_name)  
node2_addr = available_ips.pop(0)
node2_iface.ip_addr_add(addr=node2_addr, subnet=subnet)

stdout, stderr = node2.execute(f'ip addr show {node2_iface.get_device_name()}')

## Run the Experiment

We will find the ping round trip time for this pair of sites.  Your experiment should be more interesting!


In [ ]:
node1 = slice.get_node(name=node1_name)        

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [7]:
slice = fablib.get_slice(name=slice_name)
slice.delete()